---
title: "Closing the health gap across the Mediterranean, 1952 to 2025"
subtitle: "Comparing North Africa and Southern Europe with dot plots, slope charts, small multiples and normalized views"
author:
  - name: Souheil Mouhajir
    email: souheil.mouhajir@gwmail.gwu.edu
    affiliation:
      - name: The George Washington University, Elliott School of International Affairs
        city: Washington, DC
date: "2026-10-02"
date-modified: "2026-10-01"
date-format: long
language:
  title-block-published: "Published"
  title-block-modified: "Created"
abstract: |
  This notebook compares life expectancy at birth in five North African countries (Morocco, Algeria, Tunisia, Libya, Egypt) and five Southern European countries (Spain, Portugal, Italy, Greece, France) from 1952 to 2025, using the Gapminder Fast Track dataset. The median gap between the two groups fell from 24.6 years in 1952 to 7.5 years in 2002 and has not narrowed since: it was 8.9 years in 2025. Every North African country gained at least 28.6 years over the period, against 11.3 to 21.7 years in Southern Europe. Income did not drive the convergence. GDP per capita indexed to 1952 reached a median of 557 in North Africa and 918 in Southern Europe. The slope chart gives the clearest answer to the comparison question, and the small multiples show when the convergence stopped.
keywords:
  - life expectancy
  - convergence
  - North Africa
  - Southern Europe
  - Gapminder
  - slope chart
  - small multiples
  - index to baseline
format:
  html:
    theme:
      light: [cosmo, theme.scss]
      dark: [superhero, theme-dark.scss]
    html-math-method: plain
    embed-resources: true
    toc: true
    code-fold: show
jupyter: python3
---

## Comparison question

Did North Africa close the life expectancy gap with Southern Europe between 1952 and 2025, and did income follow the same path?

The data is the Gapminder **Fast Track** dataset ([open-numbers/ddf\-\-gapminder\-\-fasttrack](https://github.com/open-numbers/ddf--gapminder--fasttrack) on GitHub), which publishes annual life expectancy at birth (`lex`, version 15, April 5, 2026) and GDP per capita in PPP at constant 2021 international dollars (`gdp_pcap`, version 32, September 20, 2025). Two groups of five countries each are compared. North Africa is Morocco, Algeria, Tunisia, Libya and Egypt. Southern Europe is Spain, Portugal, Italy, Greece and France. The notebook keeps one year in five from 1952 to 2022, plus 2025, which gives 16 time points. The earliest (1952) and latest (2025) give the two conditions for the slope chart.

::: {.callout-note appearance="simple" collapse=true}
## Data source

Life expectancy: <https://raw.githubusercontent.com/open-numbers/ddf--gapminder--fasttrack/master/countries_etc_datapoints/ddf--datapoints--lex--by--country--time.csv>

GDP per capita: <https://raw.githubusercontent.com/open-numbers/ddf--gapminder--fasttrack/master/countries_etc_datapoints/ddf--datapoints--gdp_pcap--by--country--time.csv>

A snapshot of both files, cut to the ten countries and to 1950 onward, is in the `data` folder of this submission. The 2025 values are the most recent year in the series and Gapminder revises recent years between versions, so a later download may differ slightly.
:::

## Reproducibility

The notebook reads the snapshot in `data/` when it is present and downloads the two files from GitHub only when it is missing, so the render does not depend on the network. The packages are pinned in `requirements.txt` (`pip install -r requirements.txt`). The cell below prints the versions used for this render.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import plotly
import plotly.express as px
import plotly.io as pio
import scipy
from scipy import stats
print("python", sys.version.split()[0])
print("pandas", pd.__version__)
print("matplotlib", matplotlib.__version__)
print("plotly", plotly.__version__)
print("scipy", scipy.__version__)

In [ ]:
base_url = "https://raw.githubusercontent.com/open-numbers/ddf--gapminder--fasttrack/master/countries_etc_datapoints/"
codes = {"mar": "Morocco", "dza": "Algeria", "tun": "Tunisia", "lby": "Libya", "egy": "Egypt", "esp": "Spain", "prt": "Portugal", "ita": "Italy", "grc": "Greece", "fra": "France"}
os.makedirs("data", exist_ok=True)
for var in ["lex", "gdp_pcap"]:
    path = f"data/{var}.csv"
    if not os.path.exists(path):
        full = pd.read_csv(f"{base_url}ddf--datapoints--{var}--by--country--time.csv")
        full = full[full["country"].isin(codes) & (full["time"] >= 1950)]
        full.to_csv(path, index=False)
lex = pd.read_csv("data/lex.csv")
gdp = pd.read_csv("data/gdp_pcap.csv")
print(lex.shape, gdp.shape)

In [ ]:
#| output: asis
years = list(range(1952, 2023, 5)) + [2025]
df = lex.merge(gdp, on=["country", "time"])
df = df[df["time"].isin(years)].copy()
df["country"] = df["country"].map(codes)
df = df.rename(columns={"time": "year", "lex": "lifeExp", "gdp_pcap": "gdpPercap"})
north_africa = ["Morocco", "Algeria", "Tunisia", "Libya", "Egypt"]
df["group"] = "Southern Europe"
df.loc[df["country"].isin(north_africa), "group"] = "North Africa"
colors = {"North Africa": "darkorange", "Southern Europe": "steelblue"}
print(df["year"].nunique(), "time points,", df["country"].nunique(), "countries")
print()
print(df.head().to_markdown(index=False, floatfmt=".1f"))

In [ ]:
#| output: asis
medians = df.groupby(["year", "group"])["lifeExp"].median().unstack()
medians["gap"] = medians["Southern Europe"] - medians["North Africa"]
print(medians.round(1).to_markdown(floatfmt=".1f"))

The median gap was 24.6 years in 1952. It fell fastest between 1977 and 1987, from 17.2 to 9.6 years, and reached its low of 7.5 years in 2002. Since then it has not closed further: it was 8.9 years in 2025.

## Dot plot: life expectancy in 2025

The dot plot replaces a grouped bar chart. Bars would start at zero and spend most of their length on the 0 to 70 range that no country falls in, so the 13 year spread between Egypt and Spain would look small. Countries are sorted by value, which puts the two groups in two clean blocks and makes the ranking readable without a legend lookup.

In [ ]:
d25 = df[df["year"] == 2025].sort_values("lifeExp")
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.hlines(d25["country"], 68, d25["lifeExp"], color="lightgrey", linewidth=1)
for name, sub in d25.groupby("group"):
    ax.scatter(sub["lifeExp"], sub["country"], color=colors[name], s=60, label=name, zorder=3)
for i, row in enumerate(d25.itertuples()):
    ax.text(row.lifeExp + 0.3, i, f"{row.lifeExp:.1f}", va="center", fontsize=9)
ax.set_xlim(68, 86)
ax.set_xlabel("Life expectancy at birth, 2025 (years)")
ax.set_title("No North African country reached the lowest Southern European level in 2025")
ax.annotate("Algeria, the highest in North Africa,\nis 4.9 years below Greece", xy=(76.4, 4), xytext=(69, 7), fontsize=9, arrowprops=dict(arrowstyle="->", color="grey"))
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, loc="lower right")
plt.tight_layout()
plt.show()

## Slope chart: 1952 against 2025

In [ ]:
ends = df[df["year"].isin([1952, 2025])].pivot(index="country", columns="year", values="lifeExp")
ends["group"] = df.groupby("country")["group"].first()
label_y = {"Spain": 85.0, "Italy": 83.8, "France": 82.6, "Portugal": 81.4, "Greece": 80.2, "Algeria": 77.2, "Tunisia": 76.0, "Libya": 74.6, "Morocco": 73.4, "Egypt": 70.4}
fig, ax = plt.subplots(figsize=(6, 6))
for country, row in ends.iterrows():
    ax.plot([0, 1], [row[1952], row[2025]], color=colors[row["group"]], marker="o", linewidth=1.5, alpha=0.8)
    ax.text(1.04, label_y[country], country, va="center", fontsize=8)
ax.plot([0, 1], [medians.loc[1952, "North Africa"], medians.loc[2025, "North Africa"]], color="darkorange", linewidth=4, alpha=0.4)
ax.plot([0, 1], [medians.loc[1952, "Southern Europe"], medians.loc[2025, "Southern Europe"]], color="steelblue", linewidth=4, alpha=0.4)
ax.annotate("", xy=(-0.08, 41.6), xytext=(-0.08, 66.2), arrowprops=dict(arrowstyle="<->", color="black"))
ax.text(-0.11, 53.9, "gap\n24.6 yrs", ha="right", va="center", fontsize=9)
ax.annotate("", xy=(1.30, 74.0), xytext=(1.30, 82.9), arrowprops=dict(arrowstyle="<->", color="black"))
ax.text(1.33, 78.5, "gap\n8.9 yrs", ha="left", va="center", fontsize=9)
ax.set_xticks([0, 1])
ax.set_xticklabels(["1952", "2025"])
ax.set_xlim(-0.35, 1.55)
ax.set_ylim(35, 87)
ax.set_ylabel("Life expectancy at birth (years)")
ax.set_title("North African lines are steeper: the gap fell from 25 to 9 years")
ax.text(0.5, 45, "thick lines are group medians", ha="center", fontsize=8, color="grey")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

The 2025 values bunch within 2 to 3 years of each other inside each group, so the country labels on the right are spread apart by hand and keep the 2025 rank order. The dot plot above gives the exact values.

### Paired dots: the same comparison, sorted by gain

The slope chart cannot show which country gained most, because ten lines cross in a narrow band. The paired dot chart below puts the 1952 and 2025 values on one row per country and sorts the rows with `np.argsort` on the gain. The gain in years is printed at the end of each row.

In [ ]:
ends["gain"] = ends[2025] - ends[1952]
order_idx = np.argsort(ends["gain"].to_numpy())
paired = ends.iloc[order_idx]
yy = np.arange(len(paired))
fig, ax = plt.subplots(figsize=(7, 4.5))
for i, country in enumerate(paired.index):
    ax.plot([paired.loc[country, 1952], paired.loc[country, 2025]], [i, i], color=colors[paired.loc[country, "group"]], linewidth=2, alpha=0.6)
    ax.text(paired.loc[country, 2025] + 0.6, i, f"+{paired.loc[country, 'gain']:.1f}", va="center", fontsize=8)
ax.scatter(paired[1952], yy, color="white", edgecolor="black", s=40, zorder=3, label="1952")
ax.scatter(paired[2025], yy, color="black", s=40, zorder=3, label="2025")
ax.set_yticks(yy)
ax.set_yticklabels(paired.index)
ax.set_xlim(34, 89)
ax.set_xlabel("Life expectancy at birth (years)")
ax.set_title("Algeria gained the most (+39.1 years), Greece the least (+11.3)")
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, loc="lower left")
plt.tight_layout()
plt.show()

The sort splits the two groups cleanly: the smallest North African gain, Egypt at 28.6 years, is still 6.9 years above the largest Southern European gain, Portugal at 21.7.

## Small multiples on a shared y scale

Each panel shows one country, with the other group's median in grey for reference. All panels share the y axis from 35 to 86 years, so a steeper climb in one panel is a steeper climb in fact. The x axis has 16 time points, which would crowd a 1.5 inch panel, so the ticks follow the first, middle and last rule from class: 1952, 1992 and 2025. The y ticks are cut to three values for the same reason, and only the left column shows them. Within each row, countries are ordered by their 2025 value.

In [ ]:
order = ["Egypt", "Morocco", "Libya", "Tunisia", "Algeria", "Greece", "Portugal", "France", "Italy", "Spain"]
fig, axes = plt.subplots(2, 5, figsize=(11, 5), sharex=True, sharey=True)
for ax, country in zip(axes.flat, order):
    sub = df[df["country"] == country].sort_values("year")
    grp = sub["group"].iloc[0]
    other = "Southern Europe" if grp == "North Africa" else "North Africa"
    ax.plot(medians.index, medians[other], color="lightgrey", linewidth=2)
    ax.plot(sub["year"], sub["lifeExp"], color=colors[grp], linewidth=2)
    ax.set_title(country, fontsize=10)
    ax.set_xticks([1952, 1992, 2025])
    ax.set_yticks([40, 60, 80])
    ax.set_ylim(35, 86)
    ax.spines[["top", "right"]].set_visible(False)
axes[0, 0].set_ylabel("North Africa\nlife expectancy")
axes[1, 0].set_ylabel("Southern Europe\nlife expectancy")
axes[0, 0].text(1985, 40, "grey: other\ngroup median", fontsize=8, color="grey")
fig.suptitle("Every North African country gained 28 or more years; Southern Europe gained 11 to 22", fontsize=11)
plt.tight_layout()
plt.show()

## Normalized view: income indexed to 1952

The index divides each country's GDP per capita by its 1952 value and multiplies by 100. This removes the level difference and shows growth alone, on a log scale so that equal vertical distances are equal growth rates.

In [ ]:
label_pos = {"Portugal": 1080, "Egypt": 870, "Algeria": 527, "Morocco": 360}
start = df[df["year"] == 1952].set_index("country")["gdpPercap"]
df["gdp_index"] = 100 * df["gdpPercap"] / df["country"].map(start)
fig, ax = plt.subplots(figsize=(7.5, 4.5))
for country, sub in df.sort_values("year").groupby("country"):
    grp = sub["group"].iloc[0]
    ax.plot(sub["year"], sub["gdp_index"], color=colors[grp], linewidth=1.2, alpha=0.7)
    last = sub[sub["year"] == 2025]["gdp_index"].iloc[0]
    if country in label_pos:
        ax.text(2026, label_pos[country], f"{country} {last:.0f}", va="center", fontsize=8)
ax.annotate("Libya: 906 in 2007,\n459 in 2022", xy=(2022, 459), xytext=(2027, 200), fontsize=8, color="darkorange", arrowprops=dict(arrowstyle="->", color="grey"))
ax.annotate("Greece: 965 in 2007,\n724 in 2012", xy=(2012, 724), xytext=(1985, 1150), fontsize=8, color="steelblue", arrowprops=dict(arrowstyle="->", color="grey"))
ax.axhline(100, color="black", linewidth=0.8, linestyle="--")
ax.set_yscale("log")
ax.set_ylim(80, 1500)
ax.minorticks_off()
ax.set_yticks([100, 200, 400, 800])
ax.set_yticklabels(["100", "200", "400", "800"])
ax.set_xticks([1952, 1972, 1992, 2012, 2025])
ax.set_xlim(1950, 2040)
ax.set_ylabel("GDP per capita, index (1952 = 100, log scale)")
ax.set_title("Egypt's income grew as fast as Spain's; Morocco's grew the least")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

Morocco ended at 360, the lowest in the set, and Portugal at 990, the highest. Egypt (935) matched Spain (937). The group medians were 557 for North Africa and 918 for Southern Europe, so income grew more slowly in North Africa while its life expectancy grew faster.

### Annual rate of change between time points

The second normalized view is the percent change in median life expectancy between consecutive time points, using `pct_change` logic. The last interval runs 2022 to 2025 (3 years) where the others run 5, so each change is converted to an annual rate before plotting.

In [ ]:
pairs = medians[["North Africa", "Southern Europe"]]
gaps = pd.Series(medians.index, index=medians.index).diff()
ratio = pairs / pairs.shift(1)
rate = (ratio.pow(1 / gaps, axis=0) - 1) * 100
fig, ax = plt.subplots(figsize=(7.5, 4))
for name in rate.columns:
    ax.plot(rate.index, rate[name], color=colors[name], marker="o", linewidth=1.5, label=name)
ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_xticks([1957, 1982, 2002, 2025])
ax.set_ylim(-0.6, 1.95)
ax.set_ylabel("Annual % change in median\nlife expectancy")
ax.set_title("North Africa's lead in pace ended after 2002")
ax.annotate("1977 to 1982: +1.76% a year", xy=(1982, 1.76), xytext=(1988, 1.6), fontsize=9, arrowprops=dict(arrowstyle="->", color="grey"))
ax.annotate("2017 to 2022: Southern Europe\nfell 0.12% a year", xy=(2022, -0.12), xytext=(1992, -0.45), fontsize=8, arrowprops=dict(arrowstyle="->", color="grey"))
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, loc="upper right")
plt.tight_layout()
plt.show()

The North African median grew faster in every interval up to 2002, with a peak of 1.76 percent a year from 1977 to 1982. After 2002 Southern Europe was level or ahead in every interval except 2017 to 2022, when its own median fell. That is why the gap stopped narrowing.

## Extension: group means with 95 percent confidence intervals

The medians above hide how spread out each group is. This chart plots the group mean for each year as a line with a shaded 95 percent confidence interval from the t distribution (n = 5 countries, 4 degrees of freedom). With only five countries per group the intervals are wide, between 1.0 and 4.8 years, so the test is whether they separate. They never overlap at any of the 16 time points.

In [ ]:
summary = df.groupby(["group", "year"])["lifeExp"].agg(["mean", "sem"]).reset_index()
summary["half_width"] = summary["sem"] * stats.t.ppf(0.975, 4)
fig, ax = plt.subplots(figsize=(7.5, 4.5))
for name, sub in summary.groupby("group"):
    ax.fill_between(sub["year"], sub["mean"] - sub["half_width"], sub["mean"] + sub["half_width"], color=colors[name], alpha=0.2)
    ax.plot(sub["year"], sub["mean"], color=colors[name], marker="o", linewidth=1.5, label=name)
ax.set_xticks([1952, 1972, 1992, 2012, 2025])
ax.set_ylim(33, 88)
ax.set_ylabel("Mean life expectancy (years), 95% CI")
ax.set_title("The two bands move closer but never overlap")
ax.annotate("2025: 74.0 (71.1 to 77.0)\nagainst 82.7 (81.6 to 83.8)", xy=(2025, 77.0), xytext=(1990, 50), fontsize=9, arrowprops=dict(arrowstyle="->", color="grey"))
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, loc="lower right")
plt.tight_layout()
plt.show()

In [ ]:
#| output: asis
print(summary[summary["year"].isin([1952, 1987, 2025])].to_markdown(index=False, floatfmt=".1f"))

## Extension: interactive income and life expectancy by year

The Flourish race below ranks countries on life expectancy alone. This `plotly.express` chart adds the second variable in the comparison question, income, in the format Gapminder made standard: GDP per capita on a log x axis against life expectancy on the y axis, one point per country, animated across the 16 time points. Both axes are fixed for every frame (GDP per capita from $1,500 to $100,000, life expectancy from 35 to 88 years), so movement on the chart is real change and not a rescaled axis. The slider or the play button steps through the years, and hovering on a point shows the country, year and both values.

In [ ]:
#| output: asis
bubble = df.sort_values(["year", "country"])
fig = px.scatter(bubble, x="gdpPercap", y="lifeExp", color="group", text="country", animation_frame="year", animation_group="country", color_discrete_map=colors, log_x=True, range_x=[1500, 100000], range_y=[35, 88], hover_data={"gdpPercap": ":,.0f", "lifeExp": ":.1f", "group": False}, labels={"gdpPercap": "GDP per capita, PPP (constant 2021 international $, log scale)", "lifeExp": "Life expectancy at birth (years)", "group": ""}, title="Income and life expectancy, 1952 to 2025", height=560)
fig.update_traces(marker_size=14, textposition="top center")
print("```{=html}")
print(fig.to_html(include_plotlyjs=True, full_html=False, auto_play=False))
print("```")

The North African points climb almost straight up while moving little to the right: Morocco gained 31.8 years of life expectancy while its income index rose to 360. The Southern European points move right more than up. Egypt is the exception in North Africa, with an income index of 935 by 2025, level with Spain's 937, yet it has the lowest life expectancy of the ten countries, 70.4 years.

## Extension: Flourish bar chart race

The bar chart race ranks the ten countries on life expectancy for every year from 1952 to 2025, with North Africa in orange and Southern Europe in blue. The two groups start in separate blocks: in 1952 no North African country is within 15 years of Portugal, the lowest in Southern Europe. North African bars then gain on the European ones through the 1970s and 1980s, the decade in which the median gap fell from 17.2 to 9.6 years. No North African bar passes a European one in any year. After 2002 the median gap holds between 7.5 and 8.9 years. The captions mark the turning points in the gap.

```{=html}
<div style="width:100%;max-width:900px;margin:0 auto;">
<iframe data-external="1" src="https://flo.uri.sh/visualisation/30430360/embed" title="Flourish bar chart race of life expectancy, 1952 to 2025" class="flourish-embed-iframe" frameborder="0" scrolling="no" style="width:100%;height:650px;border:none;" sandbox="allow-same-origin allow-forms allow-scripts allow-downloads allow-popups allow-popups-to-escape-sandbox allow-top-navigation-by-user-activation"></iframe>
</div>
```

## Discussion

The slope chart best answers the question. The angle of each line is the gain, and the two gap markers give the convergence as one number per end point, 24.6 years in 1952 and 8.9 in 2025. Every orange line is steeper than every blue line. The cost is that it hides the path between the two years.

That path matters here, and the small multiples recover it. Most of the convergence came in 15 years: Algeria gained 14.6 years between 1972 and 1987, while Spain gained 3.9. The panels also show the North African lines flattening after 2002, which the slope chart cannot. The shared y scale makes panels comparable, and the thinned ticks keep ten panels legible.

The dot plot is the clearest ranking for one year but says nothing about change. The paired dot chart adds the gain.

The annual rate chart dates the stall: North Africa's lead in pace ended after 2002, and the gap has stayed between 7.5 and 8.9 years since. The income index shows that convergence did not come from income: the North African median index was 557 against 918 in Southern Europe.

Source: Gapminder Foundation, Fast Track dataset, `lex` v15 (April 5, 2026) and `gdp_pcap` v32 (September 20, 2025), [open-numbers/ddf\-\-gapminder\-\-fasttrack](https://github.com/open-numbers/ddf--gapminder--fasttrack) on GitHub.